# QLoRA Fine-Tuning — Qwen2.5-3B-Instruct on NVIDIA 10-K Q&A

Fine-tunes `Qwen/Qwen2.5-3B-Instruct` with QLoRA on the financial-analyst Q&A dataset
built in Phase 2 (`data/train.jsonl` / `data/eval.jsonl`). Designed to run on a free-tier
Colab **T4** GPU, with checkpoints saved directly to Google Drive so training can resume
if the runtime disconnects.

**Before running:** upload `train.jsonl` and `eval.jsonl` to a folder in your Google Drive
(default expected path: `MyDrive/financial-analyst-agent/data/`).

## 1. Environment setup

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/financial-analyst-agent'
DATA_DIR = f'{DRIVE_ROOT}/data'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
ADAPTER_DIR = f'{DRIVE_ROOT}/adapter'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU detected — set Runtime > Change runtime type > T4 GPU'

device_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {device_name}')
print(f'Total memory: {total_mem_gb:.1f} GB')

if 'T4' not in device_name:
    print(f'WARNING: expected a T4 runtime, got "{device_name}" — memory/batch settings below assume T4.')

## 2. Load base model in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False

## 3. Load train/eval datasets

In [ ]:
from datasets import load_dataset

data_files = {
    'train': f'{DATA_DIR}/train.jsonl',
    'eval': f'{DATA_DIR}/eval.jsonl',
}

# Fallback: if the files aren't in Drive yet, upload them directly to this Colab session.
if not os.path.exists(data_files['train']):
    print('train.jsonl not found in Drive — upload train.jsonl and eval.jsonl now:')
    from google.colab import files
    uploaded = files.upload()
    data_files = {'train': 'train.jsonl', 'eval': 'eval.jsonl'}

dataset = load_dataset('json', data_files=data_files)
print(dataset)
print(dataset['train'][0])

## 4. LoRA configuration

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.3f}% of {total_params:,} total)')

## 5. Trainer setup

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,
    eval_strategy='epoch',
    save_strategy='steps',
    save_steps=10,
    save_total_limit=3,
    logging_steps=5,
    max_seq_length=2048,
    packing=False,
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    processing_class=tokenizer,
)

## 6. Train (checkpoints save directly to Drive)

`output_dir` above already points at `CHECKPOINT_DIR` inside Drive, so every checkpoint
(every `save_steps=10` steps) is written straight to Drive — not just Colab's local disk —
so nothing is lost if the runtime disconnects mid-run.

In [ ]:
import glob

existing_checkpoints = sorted(
    glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda p: int(p.rsplit('-', 1)[-1]),
)
resume_from = existing_checkpoints[-1] if existing_checkpoints else None
print(f'Resuming from: {resume_from}' if resume_from else 'Starting fresh training run.')

In [ ]:
trainer.train(resume_from_checkpoint=resume_from)

If the runtime disconnects, just re-run the notebook from the top — the cell above will
automatically pick up the latest checkpoint in Drive and resume from there.

## 7. Save final adapter to Drive

In [ ]:
# Saves only the LoRA adapter weights (not the full base model)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')
!ls -la {ADAPTER_DIR}

## 8. Zip adapter and download

In [ ]:
import shutil

zip_path = shutil.make_archive('/content/qwen25_3b_nvda_lora_adapter', 'zip', ADAPTER_DIR)
print(f'Zipped adapter: {zip_path}')

from google.colab import files
files.download(zip_path)